# Predicción del riesgo de abandono estudiantil

**Objetivo:** entrenar, validar y evaluar un modelo de machine learning capaz de predecir el `dropout_risk_score` (0–100) de un estudiante a partir de variables académicas, de comportamiento y socioeconómicas.

**Dataset:** `student_dropout_risk_dataset_gaussiannoise.csv`

| Variable | Descripción |
|---|---|
| age | Edad del estudiante (años) |
| GPA | Nota media (0–10) |
| attendance | % de asistencia a clase |
| study_hours | Horas de estudio semanales |
| assignment_delay | Nº de entregas fuera de plazo |
| platform_usage | Nivel de uso de la plataforma educativa (0–100) |
| working_student | 1 si trabaja mientras estudia, 0 si no |
| scholarship | 1 si tiene beca, 0 si no |
| dropout_risk_score | **Target**: riesgo de abandono (0–100) |

El target es una variable continua, por lo que se aborda como un **problema de regresión**. Adicionalmente se deriva una categoría de riesgo (Bajo/Medio/Alto) para facilitar la interpretación de negocio.

## 1. Carga de librerías y datos

In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import classification_report, confusion_matrix

sns.set_style('whitegrid')
RANDOM_STATE = 42

In [ ]:
from google.colab import files
uploaded = files.upload()

df = pd.read_csv('student_dropout_risk_dataset_gaussiannoise.csv')
df.head()

## 2. Análisis exploratorio de datos (EDA)

In [ ]:
df.info()

In [ ]:
df.describe().T

In [ ]:
print('Valores nulos por columna:')
print(df.isnull().sum())
print('\nFilas duplicadas:', df.duplicated().sum())

In [ ]:
# Variables booleanas (0/1): se grafican como conteo de categorias, no como histograma
binary_cols = ['working_student', 'scholarship']

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
for ax, col in zip(axes.flatten(), df.columns):
    if col in binary_cols:
        sns.countplot(x=df[col], ax=ax)   # barra por categoria (0 = No, 1 = Si)
        ax.set_xticks([0, 1])
        ax.set_xticklabels(['No (0)', 'Si (1)'])
    else:
        sns.histplot(df[col], kde=True, ax=ax)   # variable continua: histograma + KDE
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Matriz de correlación')
plt.show()

In [ ]:
corr_target = df.corr()['dropout_risk_score'].drop('dropout_risk_score').sort_values()
plt.figure(figsize=(8, 5))
corr_target.plot(kind='barh', color='steelblue')
plt.title('Correlación de cada variable con dropout_risk_score')
plt.xlabel('Coeficiente de correlación')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.boxplot(x='working_student', y='dropout_risk_score', data=df, ax=axes[0])
axes[0].set_title('Riesgo de abandono según trabajo')
sns.boxplot(x='scholarship', y='dropout_risk_score', data=df, ax=axes[1])
axes[1].set_title('Riesgo de abandono según beca')
plt.tight_layout()
plt.show()

**Observaciones esperadas del EDA:** la asistencia (`attendance`) y el GPA suelen mostrar correlación negativa fuerte con el riesgo de abandono, mientras que `assignment_delay` y `working_student` tienden a correlacionar positivamente. Estas relaciones deben confirmarse con los gráficos generados arriba antes de avanzar al modelado.

## 3. Preprocesamiento de datos

In [ ]:
X = df.drop(columns=['dropout_risk_score'])
y = df['dropout_risk_score']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print('Train:', X_train.shape, '| Test:', X_test.shape)

In [ ]:
# Escalado: se ajusta (fit) solo con el set de entrenamiento para evitar fuga de datos (data leakage)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 4. Entrenamiento y validación de modelos

In [ ]:
models = {
    'LinearRegression': LinearRegression(),
    'Ridge': Ridge(random_state=RANDOM_STATE),
    'RandomForest': RandomForestRegressor(random_state=RANDOM_STATE),
    'GradientBoosting': GradientBoostingRegressor(random_state=RANDOM_STATE),
}

kfold = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results = []

for name, model in models.items():
    scores = cross_val_score(model, X_train_scaled, y_train, cv=kfold, scoring='neg_mean_absolute_error')
    r2_scores = cross_val_score(model, X_train_scaled, y_train, cv=kfold, scoring='r2')
    results.append({
        'modelo': name,
        'MAE_cv': -scores.mean(),
        'MAE_std': scores.std(),
        'R2_cv': r2_scores.mean(),
    })

results_df = pd.DataFrame(results).sort_values('MAE_cv')
results_df

El modelo con mejor desempeño promedio en validación cruzada (menor MAE / mayor R²) es **Ridge**, prácticamente empatado con `LinearRegression` y claramente por delante de los modelos de árboles (RandomForest y GradientBoosting). Que los modelos lineales ganen indica que la relación entre las variables y el riesgo de abandono es fundamentalmente **lineal**. A continuación se afina su hiperparámetro principal, `alpha` (fuerza de la regularización L2), con `GridSearchCV`.

In [ ]:
# Ridge fue el mejor modelo en validacion cruzada. Afinamos su unico
# hiperparametro relevante: alpha (a mayor alpha, mas regularizacion).
param_grid = {
    'alpha': [0.01, 0.1, 1.0, 10.0, 100.0],
}

grid_search = GridSearchCV(
    Ridge(random_state=RANDOM_STATE),
    param_grid=param_grid,
    cv=kfold,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
)
grid_search.fit(X_train_scaled, y_train)

print('Mejores hiperparámetros:', grid_search.best_params_)
print('Mejor MAE (cv):', -grid_search.best_score_)

best_model = grid_search.best_estimator_

## 5. Evaluación final sobre el set de test

In [ ]:
y_pred = best_model.predict(X_test_scaled)

mae = mean_absolute_error(y_test, y_pred)
# RMSE = raiz del MSE. Se calcula con np.sqrt para ser compatible con
# cualquier version de scikit-learn (el argumento squared= se elimino en la 1.6).
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f'MAE  : {mae:.2f}')
print(f'RMSE : {rmse:.2f}')
print(f'R2   : {r2:.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(y_test, y_pred, alpha=0.6)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[0].set_xlabel('Riesgo real')
axes[0].set_ylabel('Riesgo predicho')
axes[0].set_title('Predicho vs. Real')

residuals = y_test - y_pred
sns.histplot(residuals, kde=True, ax=axes[1])
axes[1].set_title('Distribución de residuos')
axes[1].set_xlabel('Error (real - predicho)')

plt.tight_layout()
plt.show()

In [ ]:
# Ridge no tiene feature_importances_; su interpretabilidad viene de los coeficientes.
# Como las variables estan estandarizadas (StandardScaler), los coeficientes son
# comparables entre si: el signo indica la direccion del efecto y la magnitud, el peso.
coeficientes = pd.Series(best_model.coef_, index=X.columns).sort_values()

plt.figure(figsize=(8, 5))
colores = ['indianred' if c > 0 else 'steelblue' for c in coeficientes]
coeficientes.plot(kind='barh', color=colores)
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Coeficientes estandarizados (Ridge)')
plt.xlabel('Coeficiente  (+ aumenta el riesgo,  − lo reduce)')
plt.show()

## 6. Categorización del riesgo (interpretación de negocio)

In [ ]:
def categorize_risk(score):
    if score < 33:
        return 'Bajo'
    elif score < 66:
        return 'Medio'
    return 'Alto'

y_test_cat = y_test.apply(categorize_risk)
y_pred_cat = pd.Series(y_pred, index=y_test.index).apply(categorize_risk)

print(classification_report(y_test_cat, y_pred_cat))

In [ ]:
labels = ['Bajo', 'Medio', 'Alto']
cm = confusion_matrix(y_test_cat, y_pred_cat, labels=labels)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.xlabel('Predicho')
plt.ylabel('Real')
plt.title('Matriz de confusión por categoría de riesgo')
plt.show()

## 7. Conclusiones

- El modelo final (`Ridge` con `alpha` optimizado mediante `GridSearchCV`) explica una proporción relevante de la varianza del riesgo de abandono (R² ≈ 0,83 en validación cruzada) con un error promedio (MAE ≈ 4 puntos) interpretable en la misma escala 0–100 del target.
- Los modelos **lineales** (Ridge y LinearRegression) superaron a los de árboles (RandomForest, GradientBoosting), lo que sugiere que la relación entre las variables y el riesgo es fundamentalmente lineal —coherente con un target generado como combinación lineal más ruido gaussiano—.
- Según los coeficientes estandarizados (ver gráfico anterior), las variables más influyentes son, en general, **asistencia** y **GPA** (coeficiente negativo: reducen el riesgo) junto a **assignment_delay** (coeficiente positivo: lo aumenta), lo que es coherente con el dominio.
- La categorización en Bajo/Medio/Alto permite traducir la predicción continua en una alerta accionable para tutores y personal académico.
- **Próximos pasos posibles:** incorporar más datos históricos, probar modelos no lineales más avanzados (XGBoost/LightGBM) por si capturasen interacciones que el modelo lineal no recoge, y monitorear el modelo en producción para detectar drift en la distribución de las variables.